In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

In [9]:
df = pd.read_csv("/accounts/projects/yss/stephen.lu/peint-workspace/main/data/wyatt/subs/peint_df.csv.gz", compression='gzip')
df = df[df['sample_id'].isin(['d1', 'd2', 'd3'])]
# df = df[df['parent_name'] == 'naive']
df = df[df['child_name'].str.endswith("contig_h")]
print(df.shape)
print(df[['parent_heavy_aa', 'parent_light_aa']].head())
print(df[['child_heavy_aa', 'child_light_aa']].head())

(102673, 15)
                                      parent_heavy_aa  \
2   QVQLVQSGSELKRPGASVKVSCKASGYTFTSHPINWVRQVPGQGLE...   
3   QVQLVQSGSELKRPGASVKVSCKASGYTFTSHPINWVRQVPGQGLE...   
4   QVQLVQSGSELKRPGASVKVSCKASGYTFTSHPINWVRQVPGQGLE...   
8   EVQLVESGGGLVQPGGSLRLSCAASGSIFSSHWMHWVRQAPGKGLV...   
15  QVQLVESGGGVVQPGMSLRLSCAVSGLTFSSSGMHWVRQAPGKGLE...   

                                      parent_light_aa  
2   DIQMTQSPSSLSASVGDRVTVTCRASQSISHYLNWYQQTPGKAPKL...  
3   DIQMTQSPSSLSASVGDRVTVTCRASQSISHYLNWYQQTPGKAPKL...  
4   DIQMTQSPSSLSASVGDRVTVTCRASQSISHYLNWYQQTPGKAPKL...  
8   DIVMTQSPLSLPVTPGEPASISCRSSQSLLHSNGDTYLGWYLQKPG...  
15  EIVLTQSPATLSLSPGERATLSCRASQSVSTYLAWYQQKPGQAPRL...  
                                       child_heavy_aa  \
2   QVQLVQSGSELKRPGASVKVSCKTSGYTFTSHPINWVRQVPGQGLE...   
3   QVQLVQSGSELKRPGASVKVSCKASGYTFTSHPIIWVRQVPGQGLE...   
4   QVQLVQSGSELKRPGASVKVSCKASGYTFTSHPINWVRQVPGQGLE...   
8   EVQLVESGGGLVQPGGSLRLSCAASGSIFSSHWMHWVRQAPGKGLV...   
15  QVQLVESGGGVVQPGMSLR

In [10]:
# randomly sample 500 child heavy and light sequences and save them to a csv file with columns 'fv_heavy_aho' and 'fv_light_aho'
df_sample = df.sample(n=500, random_state=42)
df_sample = df_sample[['child_heavy_aa', 'child_light_aa']]
df_sample.columns = ['fv_heavy_aho', 'fv_light_aho']
df_sample.to_csv("/scratch/users/stephen.lu/projects/protevo/paper/antibody_samples/train_data_samples.csv", index=False)

In [12]:
def remove_gaps(seq: str) -> str:
    return seq.replace('-', '')

df_aho = pd.read_csv("/accounts/projects/yss/stephen.lu/peint-workspace/main/data/wyatt/aho/dwjs_wyatt.csv.gz", compression='gzip')
df_aho['fv_heavy'] = df_aho['fv_heavy_aho'].progress_apply(remove_gaps)
df_aho['fv_light'] = df_aho['fv_light_aho'].progress_apply(remove_gaps)

print(df_aho.shape)
print(df_aho[['fv_heavy', 'fv_light']].head())

100%|██████████| 470600/470600 [00:00<00:00, 1021345.72it/s]


(470600, 5)
                                            fv_heavy  \
0  EVQLVQSGAEVKKPGESLKISCRDSGTNFATYWIGWMRQMPGKGLE...   
1  EVQLVESGGGLVKPGGSLRLSCAATGFAFSTYSMNWVRQAPGKGLE...   
2  QVQLQESGPGLVKPSDTLSLTCAVSGSSINTGYWWGWLRQPPGKGL...   
3  QVQLQQWGAGLLKPSETLSLTCAVYGGSFGGYYWSWIRQPPGKGLE...   
4  QLQLQESGPGLVKPSETLSLTCTVSGGSISSSSYYWGWIRLPPGKG...   

                                            fv_light  
0  DIVLTQSPATLSLSPGERATLSCRASQSVGSYLAWYQQKPGQPHRL...  
1  EIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRL...  
2  QSALTQPASVSGSPGQSITISCTGTSSDVGGYNYVSWYQQHPGKAP...  
3  EIVMTQSPATLSVSPGERATLSCRASQSVSNNLAWYQQKPGQAPRL...  
4  QSALTQPASVSGSPGQSITISCTGTSSDIGTYNLVSWYQQHPGKAP...  


In [13]:
# randomly sample 10 seed sequences (matching heavy and light)
random_10_indices = np.random.choice(range(df_aho.shape[0]), size=10, replace=False)
seed_heavy_sequences = [df_aho['fv_heavy'][i] for i in random_10_indices]
seed_light_sequences = [df_aho['fv_light'][i] for i in random_10_indices]

# find the aho sequences that exactly match the selected naive seed sequences
# make sure the match is on both the heavy and light chains in the same row

seed_aho_tuples = []

for heavy_seq, light_seq in zip(seed_heavy_sequences, seed_light_sequences):
    df_aho_match = df_aho[(df_aho['fv_heavy'] == heavy_seq) & (df_aho['fv_light'] == light_seq)]
    if len(df_aho_match) > 0:
        print(f"Found {len(df_aho_match)} matches for {heavy_seq} and {light_seq}")
    else:
        continue
    # assert len(df_aho_match) == 1, f"Expected exactly one match, got {len(df_aho_match)}"
    fv_heavy_aho = df_aho_match.iloc[0]['fv_heavy_aho']
    fv_light_aho = df_aho_match.iloc[0]['fv_light_aho']
    seed_aho_tuples.append((fv_heavy_aho, fv_light_aho))

# save the seed aho tuples to a csv file
seed_df = pd.DataFrame(seed_aho_tuples, columns=['fv_heavy_aho', 'fv_light_aho'])
# seed_df.to_csv("/accounts/projects/yss/stephen.lu/peint-workspace/main/data/wyatt/aho/seed_aho_tuples.csv", index=False)

Found 1 matches for QVQLQESGPGLVKPSQTLSLTCTVSGGSISSGGYYWSWIRQHPGKGLEWIGYIYYSGSSYYNPSLKSRVTISEDTSKNQFSLKLSSVTAADTAVYYCARDQRGGLGWFDPWGQGTLVTVSS and DIQVTQSPSSLSASVGDRVTITCRASQTISTYLNWYQHKPEKAPKLLIYAASNLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQTSSTWTFGQGTKVEIK
Found 1 matches for QLQLQESGPGLVKPSETLSLTCTVSGGSISSSSYYWGWIRQPPGKGLEWIGSIYYSGSTYYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCASRIAADIDYWGQGTLVTVSS and EIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRLLIYDASNRATGIPARFSGSGSGTDFTLTISSLEPEDFAVYYCQQRSNWPLTFGGGTKVEIK
Found 1 matches for EVQLVESGGGLVQPGGSLRLSCAASGFAFSNYWMSWVRQAPGKGLEWLGNIKEDGSDQNYVDSVKGRIAISRDNAKNSLYLQMISLRADDTAVYYCAREVIGGVTDLDYWGQGTLVTVSS and QSALTQPPSASGSPGQSVTISCTGTSSDVGGHNYVSWYQQHPGKAPKLMTYEVSKRPSGVPDRFSGSKSGNTASLTVSGLQAEDEADYYCSSYAGGNNYVVFGGGTTLTVL
Found 1 matches for EVQLVESGGGLVQPGGSLRLSCGASGFTFSNYAMHWVRQAPGKGLEYVSAISSDGRSTYYANSMKGRFSISRDNSKNTLYLQMGSLRVEDMAVYYCVRGESGYDSGDYWGQGTLVTVSS and QSALTQPASVSGSPGQSITISCTGTSTDVGGYNSVSWFQQHPGKAPKLMIYAVSNRPSGVSDRFSGSKSVNTASLTISGLQAEDEAVYYCSSYTS

In [15]:
print(seed_df.shape)
print(seed_df[['fv_heavy_aho', 'fv_light_aho']].head())

(10, 2)
                                        fv_heavy_aho  \
0  QVQLQES-GPGLVKPSQTLSLTCTVSG-GSISSG---GYYWSWIRQ...   
1  QLQLQES-GPGLVKPSETLSLTCTVSG-GSISSS---SYYWGWIRQ...   
2  EVQLVES-GGGLVQPGGSLRLSCAASG-FAFSN-----YWMSWVRQ...   
3  EVQLVES-GGGLVQPGGSLRLSCGASG-FTFSN-----YAMHWVRQ...   
4  QVQLVQS-GAEVKKPGASIKVSCKASG-YTFNS-----FDINWVRQ...   

                                        fv_light_aho  
0  DIQVTQSPSSLSASVGDRVTITCRAS--QTIS------TYLNWYQH...  
1  EIVLTQSPATLSLSPGERATLSCRAS--QSVS------SYLAWYQQ...  
2  QSALTQP-PSASGSPGQSVTISCTGTS-SDVGG----HNYVSWYQQ...  
3  QSALTQP-ASVSGSPGQSITISCTGTS-TDVGG----YNSVSWFQQ...  
4  DIQMTQSPSTLSASVGDTVTITCRAS--QSIS------SWLAWYQH...  


In [16]:
# save the seed df to a csv file
seed_df.to_csv("/accounts/projects/yss/stephen.lu/peint-workspace/main/data/wyatt/aho/seed_aho_tuples.csv", index=False)